In [1]:
import pandas as pd
from langchain_ollama import ChatOllama
from langchain.chains import LLMChain, SimpleSequentialChain, SequentialChain
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser

In [2]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:1.7B",
    temperature=0.9,
    verbose=True,
    extract_reasoning=True,
)

In [3]:
df = pd.read_csv("../data/03.csv", usecols=["Product", "Review"])
df.head(10)

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...
5,L'Or Espresso Café \n,Je trouve le goût médiocre. La mousse ne tient...
6,Hervidor de Agua Eléctrico,"Está lu bonita calienta muy rápido, es muy fun..."


## LLMChain

In [4]:
prompt = ChatPromptTemplate.from_template("What is the best name to describe a company that makes {product}?")
chain = LLMChain(llm=llm, prompt=prompt)

/tmp/ipykernel_87299/3192878188.py:4: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [5]:
product = "Queen Size Sheet Set"
chain.invoke(product)

{'product': 'Queen Size Sheet Set',
 'text': '\n\nThe best name for a company that produces queen size sheet sets should **balance clarity, memorability, and relevance** to the product. Here are key considerations and suggestions:\n\n### **Key Elements to Include:**\n1. **Product Focus**: Emphasize the queen size (e.g., "Queen," "Queen Size").\n2. **Brand Identity**: Reflect the quality, comfort, or luxury of the product.\n3. **Memorability**: Keep the name short, simple, and easy to recall.\n4. **Target Audience**: Consider the demographic (e.g., parents, luxury buyers, eco-conscious consumers).\n\n### **Recommended Names:**\n1. **QueenSet**  \n   - **Why**: Direct, concise, and ties the product to the size. It’s catchy and avoids overcomplication.\n   - **Pros**: Memorable, clear, and immediately conveys the product’s purpose.\n\n2. **QueenSize Co.**  \n   - **Why**: Highlights the product size while adding a professional touch. It’s a strong brand name for a company.\n   - **Pros**:

## SimpleSequentialChain

In [6]:
prompt_1 = ChatPromptTemplate.from_template(
    """
What is the best unusual name (just a single option) to describe a company that makes {product}?.
No explanation needed, the name should be self-explaining
"""
)
prompt_2 = ChatPromptTemplate.from_template("Write a 20 words description for the following company: {company}")

chain_1 = LLMChain(llm=llm, prompt=prompt_1)
chain_2 = LLMChain(llm=llm, prompt=prompt_2)

In [7]:
overall_simple_chain = SimpleSequentialChain(
    chains=[chain_1, chain_2],
    verbose=True,
)
overall_simple_chain.invoke(product);



> Entering new SimpleSequentialChain chain...


Throne Sheet Set


KeyboardInterrupt: 

## SequentialChain

In [ ]:
prompt_1 = ChatPromptTemplate.from_template(
    "Translate the following review to english: {review}",
)
chain_1 = LLMChain(llm=llm, prompt=prompt_1, output_key="english_review")

prompt_2 = ChatPromptTemplate.from_template("Summarize the following review in 1 sentence: {english_review}")
chain_2 = LLMChain(llm=llm, prompt=prompt_2, output_key="summary")

prompt_3 = ChatPromptTemplate.from_template(
    "What language is the following review in? Output just the capitalized name of the language in English: {review}. "
)
chain_3 = LLMChain(llm=llm, prompt=prompt_3, output_key="language")

prompt_4 = ChatPromptTemplate.from_template(
    """
Write a follow up response to the following summary in the specified language.
Don't include any thinking, just plain response:

Summary: {summary}

Language: {language}
"""
)
chain_4 = LLMChain(llm=llm, prompt=prompt_4, output_key="followup_message")


main_chain = SequentialChain(
    chains=[chain_1, chain_2, chain_3, chain_4],
    input_variables=["review"],
    output_variables=["language", "english_review", "summary", "followup_message"],
    verbose=False,
)

df["Review"].apply(main_chain.invoke).apply(pd.Series).apply(lambda x: x.str.strip())

,review,language,english_review,summary,followup_message
0,I ordered a king size set. My only criticism w...,English,I ordered a king-size set. My only criticism w...,The reviewer praised the product's quality and...,The reviewer praised the product's quality and...
1,"I loved the waterproof sac, although the openi...",English,"I loved the waterproof pouch, although the ope...",The reviewer loved the waterproof pouch but fo...,The reviewer appreciated the waterproof pouch ...
2,This mattress had a small hole in the top of i...,ENGLISH,This mattress had a small hole in the top of i...,The review highlights a defective mattress wit...,The review highlights a defective mattress wit...
3,This is the best throw pillow fillers on Amazo...,ENGLISH,This is the best throw pillow fillers availabl...,This review praises the best throw pillow fill...,The review highlights that the best throw pill...
4,I loved this product. But they only seem to la...,English,"I loved this product. However, it seems to las...","""I loved this product, but it lasted only a fe...","I loved this product, but it lasted only a few..."
5,Je trouve le goût médiocre. La mousse ne tient...,French,I find the flavor mediocre. The foam doesn't h...,The review criticizes the product's mediocre f...,Le résumé critique le goût moyen et la retenti...
6,"Está lu bonita calienta muy rápido, es muy fun...",SPANISH,"It's nice and very hot, it's very functional, ...","The product is pleasant and hot, highly functi...","El producto es agréable y caliente, muy funcio..."


## Router Chain

In [ ]:
physics_template = """
You are a very smart physics professor. You are great at answering questions about physics in a concise and easy to understand manner.
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}
"""


math_template = """
You are a very good mathematician. You are great at answering math questions.
You are so good because you are able to break down hard problems into their component parts,
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{input}
"""

history_template = """
You are a very good historian. You have an excellent knowledge of and understanding of people,
events and contexts from a range of historical periods. You have the ability to think, reflect, debate, discuss and
evaluate the past. You have a respect for historical evidence and the ability to make use of it to support your explanations
and judgements.

Here is a question:
{input}
"""

computerscience_template = """
You are a successful computer scientist. You have a passion for creativity, collaboration,
forward-thinking, confidence, strong problem-solving capabilities, understanding of theories and algorithms,
and excellent communication skills. You are great at answering coding questions.
You are so good because you know how to solve a problem by describing the solution in imperative steps
that a machine can easily interpret and you know how to choose a solution that has a good balance between
time complexity and space complexity.

Here is a question:
{input}
"""

In [ ]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering physics questions",
        "prompt_template": physics_template,
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template,
    },
    {
        "name": "history",
        "description": "Good for answering history questions",
        "prompt_template": history_template,
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template,
    },
]

In [ ]:
destination_chains = {}
for info in prompt_infos:
    name = info["name"]
    prompt_template = info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)
destinations_str

'physics: Good for answering physics questions\nmath: Good for answering math questions\nhistory: Good for answering history questions\ncomputer science: Good for answering computer science questions'

In [ ]:
MULTI_PROMPT_ROUTER_TEMPLATE = """
Given a raw text input to a language model select the model prompt best suited for the input.
You will be given the names of the available prompts and a description of what the prompt is best suited for.
You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of the candidate prompts listed below.
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>
"""

<>:10: SyntaxWarning: invalid escape sequence '\ '
<>:10: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_83406/553828092.py:10: SyntaxWarning: invalid escape sequence '\ '
  "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}


In [ ]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str,
)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True,
)

In [ ]:
chain.invoke("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}

In [ ]:
chain.invoke("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


{'input': 'what is 2 + 2',
 'text': '\n\nThe question "what is 2 + 2" is a straightforward arithmetic operation. To solve it, we add the two numbers together:\n\n$$\n2 + 2 = 4\n$$\n\nThis is confirmed through multiple methods, including basic addition, number line visualization, algebraic reasoning, and logical interpretation. There are no special cases or tricks involved here, as it is a simple addition problem.\n\n**Answer:**  \n$$\n\\boxed{4}\n$$'}

In [ ]:
chain.invoke("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


{'input': 'Why does every cell in our body contain DNA?',
 'text': "\n\nEvery cell in the human body contains DNA, but there are exceptions. Here's a detailed explanation:\n\n1. **DNA as the Genetic Blueprint**: DNA (deoxyribonucleic acid) is the molecule that carries the genetic instructions for building and maintaining an organism. It contains the code for making proteins, which are essential for cellular functions, structure, and regulation. This genetic information is necessary for all cells to function properly, even if they are specialized (e.g., muscle cells, nerve cells, or blood cells).\n\n2. **Location of DNA**: \n   - In **eukaryotic cells** (which include all human cells except red blood cells), DNA is stored in the **nucleus**. \n   - In **prokaryotic cells** (e.g., bacteria), DNA is found in the **cytoploted** (a region in the cell's cytoplasm).\n\n3. **Exceptions**: \n   - **Mature red blood cells** in humans do not have a nucleus and, therefore, do not contain DNA. This